In [16]:
# --- LangChain and LLM Imports ---
from langchain_openai import ChatOpenAI 

# --- Document Loading and Vector Store ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Prompting and Document Utilities ---
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_classic.chains.summarize import load_summarize_chain

# --- Core and Output Parsers ---
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables.graph import MermaidDrawMethod

# --- LangGraph for Workflow Graphs ---
from langgraph.graph import END, StateGraph

# --- Standard Library Imports ---
from time import monotonic
from dotenv import load_dotenv
from pprint import pprint
import tiktoken
import json
import os

# --- Datasets and Typing ---
from datasets import Dataset
from typing_extensions import TypedDict
from IPython.display import display, Image
from typing import List, TypedDict

# --- RAGAS Metrics for Evaluation ---
from ragas import evaluate
from ragas.metrics.collections import (
    answer_correctness,
    faithfulness,
    answer_relevancy,
    context_recall,
    SemanticSimilarity
)

import langgraph

# --- Helper Functions ---
from helper_functions import (
    num_tokens_from_string,
    replace_t_with_space,
    replace_double_lines_with_one_line,
    split_into_chapters,
    analyse_metric_results,
    escape_quotes,
    text_wrap,
    extract_book_quotes_as_documents
)

# --- Load environment variables (e.g., API keys) ---
load_dotenv()

# --- Set environment variable for debugging (optional) ---
os.environ["PYDEVD_WARN_EVALUATION_TIMEOUT"] = "100000"

In [2]:
hp_pdf_path = "docs/hp/Harry Potter - Book 1 - The Sorcerers Stone.pdf"

In [3]:
# --- Split the PDF into chapters and preprocess the text ---
# 1. Split the PDF into chapters using the provided helper function.
#    This function takes the path to the PDF and returns a list of Document objects, each representing a chapter.
chapters = split_into_chapters(hp_pdf_path)
# 2. Clean up the text in each chapter by replacing unwanted characters (e.g., '\t') with spaces.
#    This ensures the text is consistent and easier to process downstream.
chapters = replace_t_with_space(chapters)
# 3. Print the number of chapters extracted to verify the result.
print(len(chapters))

17


In [4]:
chapters[0]

Document(metadata={'chapter': 1}, page_content="CHAPTER ONE\n \nTHE BOY WHO LIVED\n \n      \nM \nr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the last people\nyou’d expect to be involved in anything strange or mysterious, because they just\ndidn’t hold with such nonsense.\n      Mr. Dursley was the director of a firm called Grunnings, which made\ndrills. He was a big, beefy man with hardly any neck, although he did have a\nvery large mustache. Mrs. Dursley was thin and blonde and had nearly twice the\nusual amount of neck, which came in very useful as she spent so much of her\ntime craning over garden fences, spying on the neighbors. The Dursleys had a\nsmall son called Dudley and in their opinion there was no finer boy anywhere.\n      The Dursleys had everything they wanted, but they also had a secret, and\ntheir greatest fear was that somebody would discover it. They didn’t think they\ncould be

In [10]:
encoding = tiktoken.encoding_for_model("gpt-4o")
text = chapters[0].page_content
token_integers = encoding.encode(text)
token_count = len(token_integers)
print(f"Token Count: {token_count}")

Token Count: 6697


In [5]:
# --- Load and Preprocess the PDF, then Extract Quotes ---
# 1. Load the PDF using PyPDFLoader
loader = PyPDFLoader(hp_pdf_path)
document = loader.load()
# 2. Clean the loaded document by replacing unwanted characters (e.g., '\t') with spaces
document_cleaned = replace_t_with_space(document)
# 3. Extract a list of quotes from the cleaned document as Document objects
book_quotes_list = extract_book_quotes_as_documents(document_cleaned)
print(len(book_quotes_list))

1534


In [6]:
book_quotes_list[1000]

Document(metadata={}, page_content=' said Wood. "We\'ve just got to make sure we play a clean game, so Snape hasn\'t got an excuse to pick on us.')

In [7]:
azure_llm_key = os.getenv("azure_llm_key")
llm = ChatOpenAI(
    model="DeepSeek-V4-Flash",
    base_url="https://3t-ai-resource.services.ai.azure.com/openai/v1",
    api_key=azure_llm_key,
    max_tokens=2048,
    temperature=0.1,
)

In [11]:
summarization_prompt_template = """Write an extensive summary of the following:

{text}

SUMMARY:"""

# Create a PromptTemplate object using the template string.
# The input variable "text" will be replaced with the content to summarize.
summarization_prompt = PromptTemplate(
    template=summarization_prompt_template,
    input_variables=["text"]
)

In [12]:
def create_chapter_summary(chapter, llm):
    """
    Creates a summary of a chapter using a large language model (LLM).
    Args:
        chapter: A Document object representing the chapter to summarize.
    Returns:
        A Document object containing the summary of the chapter.
    """

    # Extract the text content from the chapter
    chapter_txt = chapter.page_content
    max_tokens = 16000  # Maximum token limit for the model
    verbose = False  # Set to True for more detailed output
    # Calculate the number of tokens in the chapter text
    num_tokens = num_tokens_from_string(chapter_txt, 'gpt-4o')

    # Choose the summarization chain type based on token count
    if num_tokens < max_tokens:
        # For shorter chapters, use the "stuff" chain type
        chain = load_summarize_chain(
            llm,
            chain_type="stuff",
            prompt=summarization_prompt,
            verbose=verbose
        )
    else:
        # For longer chapters, use the "map_reduce" chain type
        chain = load_summarize_chain(
            llm,
            chain_type="map_reduce",
            map_prompt=summarization_prompt,
            combine_prompt=summarization_prompt,
            verbose=verbose
        )
    start_time = monotonic()
    doc_chapter = Document(page_content=chapter_txt)
    summary_result = chain.invoke([doc_chapter])
    # Print chain type and execution time for reference
    print(f"Chain type: {chain.__class__.__name__}")
    print(f"Run time: {monotonic() - start_time}")
    # Clean up the summary text (remove double newlines, etc.)
    summary_text = replace_double_lines_with_one_line(summary_result["output_text"])
    # Create a Document object for the summary, preserving chapter metadata
    doc_summary = Document(page_content=summary_text, metadata=chapter.metadata)
    return doc_summary

In [14]:
# # --- Generate Summaries for Each Chapter ---
# chapter_summaries = []
# # Iterate over each chapter in the chapters list
# for chapter in chapters:
#     summary = create_chapter_summary(chapter, llm)
#     chapter_summaries.append(summary)

Chain type: StuffDocumentsChain
Run time: 12.92240441699687
Chain type: StuffDocumentsChain
Run time: 12.95165604200156
Chain type: StuffDocumentsChain
Run time: 9.6705019580113
Chain type: StuffDocumentsChain
Run time: 56.35622020800656
Chain type: StuffDocumentsChain
Run time: 7.771511041995836
Chain type: StuffDocumentsChain
Run time: 63.88727658300195
Chain type: StuffDocumentsChain
Run time: 452.6376131249999
Chain type: StuffDocumentsChain
Run time: 23.597769999993034
Chain type: StuffDocumentsChain
Run time: 10.032964792000712
Chain type: StuffDocumentsChain
Run time: 17.540286165996804
Chain type: StuffDocumentsChain
Run time: 7.838023917007376
Chain type: StuffDocumentsChain
Run time: 40.02960749999329
Chain type: StuffDocumentsChain
Run time: 31.28564929199638
Chain type: StuffDocumentsChain
Run time: 20.957695416000206
Chain type: StuffDocumentsChain
Run time: 17.759518374994514
Chain type: StuffDocumentsChain
Run time: 51.09342441699118
Chain type: StuffDocumentsChain
Run t

In [20]:
# docs_json = [doc.model_dump() for doc in chapter_summaries]
# with open("processed_docs/hp/chapter_summary.json", "w", encoding="utf-8") as f:
#     json.dump(docs_json, f, ensure_ascii=False, indent=4)

In [21]:
with open("processed_docs/hp/chapter_summary.json", "r", encoding="utf-8") as f:
    docs_json = json.load(f)
chapter_summaries = [Document(**d) for d in docs_json]

In [22]:
chapter_summaries[0]

Document(metadata={'chapter': 1}, page_content='Here is an extensive summary of Chapter One of *Harry Potter and the Sorcerer\'s Stone*.\n### Chapter One: The Boy Who Lived\nThe chapter opens by introducing the Dursley family of number four, Privet Drive. Mr. and Mrs. Dursley are a profoundly ordinary, conventional, and proud couple who despise anything strange or mysterious. Mr. Dursley is a large, beefy man who works as a director at a drill-making company, while his wife, Petunia, is thin, blonde, and spends her time spying on the neighbors. They have a spoiled son, Dudley, whom they believe to be the perfect child. The Dursleys harbor a deep, shameful secret: Petunia’s sister, Lily, and her husband, James Potter, are the very opposite of "Dursleyish." The Dursleys have cut off all contact with them, fearing what the neighbors would think of these "unDursleyish" relatives and their child, Harry.\nThe story begins on a dull, gray Tuesday. As Mr. Dursley leaves for work, he notices th

In [24]:
embedding_base_url = os.getenv("embedding_base_url")
embedding_key = os.getenv("embedding_key")
embedding_deployment = os.getenv("embedding_deployment")
embeddings = OpenAIEmbeddings(
    model=embedding_deployment,
    base_url=f"{embedding_base_url}/openai/v1",
    api_key=embedding_key,
)

In [27]:
def encode_book(path, embeddings, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a FAISS vector store using OpenAI embeddings.
    Args:
        path (str): The path to the PDF file.
        chunk_size (int): The desired size of each text chunk.
        chunk_overlap (int): The amount of overlap between consecutive chunks.
    Returns:
        FAISS: A FAISS vector store containing the encoded book content.
    """
    # 1. Load the PDF document using PyPDFLoader
    loader = PyPDFLoader(path)
    documents = loader.load()

    # 2. Split the document into chunks for embedding
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    texts = text_splitter.split_documents(documents)
    # 3. Clean up the text chunks (replace unwanted characters)
    cleaned_texts = replace_t_with_space(texts)
    # embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)
    return vectorstore

In [28]:
def encode_chapter_summaries(chapter_summaries, embeddings):
    """
    Encodes a list of chapter summaries into a FAISS vector store using OpenAI embeddings.
    Args:
        chapter_summaries (list): A list of Document objects representing the chapter summaries.
    Returns:
        FAISS: A FAISS vector store containing the encoded chapter summaries.
    """
    # Encode the chapter summaries into a FAISS vector store
    chapter_summaries_vectorstore = FAISS.from_documents(chapter_summaries, embeddings)
    # Return the vector store
    return chapter_summaries_vectorstore

In [29]:
def encode_quotes(book_quotes_list, embeddings):
    """
    Encodes a list of book quotes into a FAISS vector store using OpenAI embeddings.
    Args:
        book_quotes_list (list): A list of Document objects, each representing a quote from the book.
    Returns:
        FAISS: A FAISS vector store containing the encoded book quotes.
    """
    # Encode the book quotes into a FAISS vector store
    quotes_vectorstore = FAISS.from_documents(book_quotes_list, embeddings)
    return quotes_vectorstore

In [31]:
# --- Create or Load Vector Stores for Book Chunks, Chapter Summaries, and Book Quotes ---
# Check if the vector stores already exist on disk
if (
    os.path.exists("embedding/chunks_vector_store") and
    os.path.exists("embedding/chapter_summaries_vector_store") and
    os.path.exists("embedding/book_quotes_vectorstore")
):
    # If vector stores exist, load them using OpenAI embeddings
    chunks_vector_store = FAISS.load_local(
        "embedding/chunks_vector_store", embeddings, allow_dangerous_deserialization=True
    )
    chapter_summaries_vector_store = FAISS.load_local(
        "embedding/chapter_summaries_vector_store", embeddings, allow_dangerous_deserialization=True
    )
    book_quotes_vectorstore = FAISS.load_local(
        "embedding/book_quotes_vectorstore", embeddings, allow_dangerous_deserialization=True
    )
else:
    # If vector stores do not exist, encode and save them
    # 1. Encode the book into a vector store of chunks
    chunks_vector_store = encode_book(hp_pdf_path, embeddings, chunk_size=1000, chunk_overlap=200)

    # 2. Encode the chapter summaries into a vector store
    chapter_summaries_vector_store = encode_chapter_summaries(chapter_summaries, embeddings)

    # 3. Encode the book quotes into a vector store
    book_quotes_vectorstore = encode_quotes(book_quotes_list, embeddings)

    # 4. Save the vector stores to disk for future use
    chunks_vector_store.save_local("embedding/chunks_vector_store")
    chapter_summaries_vector_store.save_local("embedding/chapter_summaries_vector_store")
    book_quotes_vectorstore.save_local("embedding/book_quotes_vectorstore")